# 4장 실습: 전역 설명

『설명가능한 인공지능(XAI)』 4장 실습 노트북입니다. 실습 저장소: https://github.com/neomoon40/xai-textbook

1·3장과 같은 대출 심사 데이터(German Credit)와 신청자 A로, 3장의 랜덤 포레스트를 블랙박스로 두고 PDP, ICE, ALE, 순열 중요도, 전역 대리 모델을 만듭니다. 3장의 EBM은 정답지로 함께 학습합니다. 셀을 위에서부터 차례로 실행하세요. 뒤의 셀은 앞 셀이 만든 데이터와 모델을 씁니다. 코드 4-7(모델 10개의 순열 중요도)과 코드 4-8(교차검증)이 가장 오래 걸려 Colab CPU 런타임에서 각각 수 분, 1분 안팎이 걸립니다.

## 0. 라이브러리 설치와 버전 맞추기 (Colab)

이 책의 결과는 Python 3.13, scikit-learn 1.9.1, pandas 3.0.5, NumPy 2.5.3, SciPy 1.18.1, interpret 0.7.8, PyALE 1.2.0에서 얻었습니다. Colab에는 interpret와 PyALE가 없고 다른 라이브러리도 버전이 다르므로, **새 세션을 열 때마다** 아래 셀을 가장 먼저 실행합니다.

1. 아래 셀의 둘째~넷째 줄, **세 줄 모두** 맨 앞 `#`을 지우고 실행합니다.
2. 빨간 글씨로 `ERROR: pip's dependency resolver ...` 경고가 나올 수 있습니다. 마지막 줄에 `Successfully installed`가 나오면 설치는 끝난 것입니다.
3. **[런타임] → [세션 다시 시작]**을 누른 뒤, 이 셀은 건너뛰고 코드 4-1부터 실행합니다. 세션을 다시 시작해도 설치한 패키지는 남아 있습니다.

내 컴퓨터에서 부록 B의 requirements.txt로 이미 설치했다면 이 셀은 실행하지 않습니다.

In [ ]:
# Colab에서는 다음 세 줄의 #을 지우고 실행한다
# !pip install scikit-learn==1.9.1 pandas==3.0.5 numpy==2.5.3
# !pip install scipy==1.18.1 interpret==0.7.8 PyALE==1.2.0
# !pip install koreanize-matplotlib


### 실습 1. PDP와 ICE

#### 코드 4-1 준비: 데이터, 랜덤 포레스트, 정답지 EBM

코드 4-1의 버전 출력이 scikit-learn 1.9.1이 아니면 설치 셀을 실행했는지, 세션을 다시 시작했는지 확인합니다. 수치형 열을 실수형으로 바꾸는 줄을 빠뜨리면 코드 4-2에서 정수형 열 오류가 납니다. EBM의 숫자는 버전이 같아도 CPU에 따라 끝자리가 조금 다를 수 있습니다(책의 숫자는 Colab CPU 런타임 기준).

In [ ]:
import numpy as np, pandas as pd, sklearn, matplotlib.pyplot as plt
import koreanize_matplotlib            # 그래프에 한글을 쓰기 위해
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from interpret.glassbox import ExplainableBoostingClassifier
print("scikit-learn", sklearn.__version__)

df = fetch_openml("credit-g", version=1, as_frame=True).frame
X = df.drop(columns="class"); y = (df["class"] == "good").astype(int)
num = X.select_dtypes("number").columns
# scikit-learn 1.9의 PDP는 정수형 열을 받지 않으므로 실수형으로 바꾼다
X[num] = X[num].astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
A = 87; x_a = X_test.loc[[A]]   # 신청자 A

def prep(scale=False):          # 3장 코드 3-1과 같은 전처리
    cat = make_column_selector(dtype_include="category")
    num = make_column_selector(dtype_include="number")
    return make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), cat),
        (StandardScaler() if scale else "passthrough", num),
        verbose_feature_names_out=False)

def RF(**kw):                   # 나무 300그루의 랜덤 포레스트
    forest = RandomForestClassifier(n_estimators=300, **kw)
    return make_pipeline(prep(), forest)

rf = RF(random_state=42).fit(X_train, y_train)
ebm = ExplainableBoostingClassifier(random_state=42).fit(X_train, y_train)
for name, m in [("랜덤 포레스트", rf), ("EBM", ebm)]:
    acc = accuracy_score(y_test, m.predict(X_test))
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    p_a = m.predict_proba(x_a)[0, 1]
    print(f"{name:<7} 정확도 {acc:.3f}  AUC {auc:.3f}  A {p_a:.2f}")


#### 코드 4-2 대출 기간·대출 금액·예금 계좌의 PDP

In [ ]:
from sklearn.inspection import partial_dependence
from sklearn.inspection import PartialDependenceDisplay
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
PartialDependenceDisplay.from_estimator(
    rf, X_train, ["duration", "credit_amount", "checking_status"],
    categorical_features=["checking_status"], ax=ax)
plt.savefig("code4-2.png", dpi=120, bbox_inches="tight")

grid = np.array([6, 12, 18, 24, 30, 36, 48, 60.0])
pdp = partial_dependence(rf, X_train, ["duration"],
                         custom_values={"duration": grid})["average"][0]
print("대출 기간 PDP")
tab = pd.DataFrame([pdp.round(3)], columns=grid.astype(int))
print(tab.to_string(index=False))
pdc = partial_dependence(rf, X_train, ["checking_status"],
                         categorical_features=["checking_status"])
print("예금 계좌 PDP")
cat = pd.Series(pdc["average"][0], index=pdc["grid_values"][0])
print(cat.round(3).to_string())


#### 코드 4-3 ICE, 중심화 ICE, 신청자 A의 곡선

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, centered in zip(ax, [False, True]):    # 오른쪽은 중심화 ICE
    PartialDependenceDisplay.from_estimator(
        rf, X_train, ["duration"], kind="both", centered=centered,
        subsample=100, random_state=42, ax=a)       # 무작위로 고른 100명
plt.savefig("code4-3.png", dpi=120, bbox_inches="tight")

ice = partial_dependence(rf, X_train, ["duration"],
                         custom_values={"duration": grid},
                         kind="individual")["individual"][0]  # 700명 × 8
change = ice[:, -1] - ice[:, 0]               # 6개월 → 60개월의 변화
print(f"변화 평균 {change.mean():.3f}, 표준편차 {change.std():.3f}")
print(f"0.2 넘게 하락 {(change < -0.2).sum()}명, "
      f"오히려 상승 {(change > 0).sum()}명")
G = np.arange(4, 73, 1.0)
ice_a = partial_dependence(rf, x_a, ["duration"],
                           custom_values={"duration": G},
                           kind="individual")["individual"][0][0]
print("A의 곡선", {int(g): round(float(v), 3)
                  for g, v in zip(G, ice_a) if g in (12, 24, 36, 48)})
print("A가 0.5 이상인 기간", G[ice_a >= 0.5].astype(int).tolist())


### 실습 2. 외삽과 ALE

#### 코드 4-4 PDP가 만드는 가상의 신청자

In [ ]:
r = X_train.duration.corr(X_train.credit_amount)
long = X_train[X_train.duration >= 48]
lo = long.credit_amount.min()
fake = X_train[X_train.credit_amount < lo]   # 60개월로 바꾸면 가상의 사람
print(f"상관 {r:.2f}, 48개월 이상 {len(long)}명의 "
      f"최소 금액 {lo:,.0f}마르크")
print(f"그보다 적게 빌린 사람 {len(fake) / len(X_train):.1%}")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(X_train.duration, X_train.credit_amount, s=8, alpha=0.5,
           label="실제 신청자")
ax.scatter(np.full(len(fake), 60), fake.credit_amount, s=8,
           color="orange", label="60개월로 바꾼 가상 신청자")
ax.axhline(lo, ls=":", color="gray"); ax.set_yscale("log")
ax.set_xlabel("대출 기간(개월)"); ax.set_ylabel("대출 금액(마르크, 로그)")
ax.legend(fontsize=8)
plt.savefig("code4-4.png", dpi=120, bbox_inches="tight")


#### 코드 4-5 PyALE로 그린 ALE와 PDP 비교

실행하면 표 앞에 `PyALE._ALE_generic:INFO: Continuous feature detected.`라는 안내 줄이 먼저 나옵니다. 오류가 아니므로 그대로 두면 됩니다.

In [ ]:
from PyALE import ale
class ProbModel:     # PyALE는 predict()를 부르므로 승인 확률을 돌려준다
    def __init__(self, m): self.m = m
    def predict(self, Z): return self.m.predict_proba(Z)[:, 1]
res = ale(X=X_train, model=ProbModel(rf), feature=["duration"],
          grid_size=10, include_CI=False, plot=False)
q = res.index.values.astype(float)               # 분위수 구간 경계
pdq = partial_dependence(rf, X_train, ["duration"],
                         custom_values={"duration": q})["average"][0]
cmp = pd.DataFrame({"ALE": res["eff"].values,
                    "PDP(중심화)": pdq - pdq.mean(),
                    "구간 사람 수": res["size"].values},
                   index=q.astype(int))
print(cmp.round(3).to_string())
print(f"폭: ALE {np.ptp(res['eff']):.3f}, PDP {np.ptp(pdq):.3f}")


### 실습 3. 순열 중요도

#### 코드 4-6 순열 중요도: 지표, 데이터, 불순도 중요도

In [ ]:
from sklearn.inspection import permutation_importance
def pi(model, Xd, yd, scoring="accuracy", n=20):
    r = permutation_importance(model, Xd, yd, scoring=scoring,
                               n_repeats=n, random_state=42)
    return pd.DataFrame({"평균": r.importances_mean,
                         "표준편차": r.importances_std}, index=Xd.columns)
acc = pi(rf, X_test, y_test)                      # 정확도, 평가 데이터
auc = pi(rf, X_test, y_test, scoring="roc_auc")   # AUC, 평가 데이터
tr = pi(rf, X_train, y_train)                     # 정확도, 학습 데이터
# 불순도 중요도: 원-핫 열을 원래 특징으로 합친다
names = rf[0].get_feature_names_out()
orig = [next(c for c in X.columns if n == c or n.startswith(c + "_"))
        for n in names]
mdi = pd.Series(rf[-1].feature_importances_, index=names)
mdi = mdi.groupby(orig).sum()
rank = lambda s: s.rank(ascending=False).astype(int)
tab = acc.sort_values("평균", ascending=False).head(8).copy()
tab["AUC 순위"] = rank(auc["평균"])
tab["학습 순위"] = rank(tr["평균"])
tab["불순도 순위"] = rank(mdi)
print(tab.round(4).to_string())
print("학습 데이터 기준 상위 3")
print(tr["평균"].nlargest(3).round(4).to_string())


#### 코드 4-7 비슷하게 좋은 모델 10개의 중요도와 A의 승인 확률

모델 8개를 새로 학습하고 10개 모델마다 순열 중요도를 10번씩 계산하므로 수 분이 걸립니다. 출력의 RF는 랜덤 포레스트, LR은 로지스틱 회귀입니다.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
gb = make_pipeline(prep(), GradientBoostingClassifier(random_state=42))
lr = make_pipeline(prep(scale=True),
                   LogisticRegression(C=0.1, max_iter=1000))
models = {"RF(1장)": rf,              # RF = 랜덤 포레스트
          **{f"RF(시드 {s})": RF(random_state=s) for s in [0, 1, 2, 4]},
          "RF(특징 30%)": RF(max_features=0.3, random_state=42),
          "RF(특징 50%)": RF(max_features=0.5, random_state=42),
          "그래디언트 부스팅": gb,
          "LR(강한 규제)": lr,         # LR = 로지스틱 회귀
          "EBM(3장)": ebm}
cols = {"checking_status": "계좌", "duration": "기간",
        "credit_amount": "금액", "age": "나이",
        "foreign_worker": "외국인"}
rows = {}
for name, m in models.items():
    if m is not rf and m is not ebm: m.fit(X_train, y_train)
    imp = pi(m, X_test, y_test, n=10)["평균"]          # 10번 반복
    rows[name] = {"정확도": accuracy_score(y_test, m.predict(X_test)),
                  **imp[list(cols)].rename(cols),
                  "A": m.predict_proba(x_a)[0, 1]}
T = pd.DataFrame(rows).T
print(T.round(3).to_string())
for c in ["기간", "금액", "A"]:
    print(f"{c} 범위 {T[c].min():.3f} ~ {T[c].max():.3f}")


### 실습 4. 전역 대리 모델

#### 코드 4-8 전역 대리 트리와 충실도

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.tree import DecisionTreeClassifier, export_text
y_cv = cross_val_predict(RF(random_state=42), X_train, y_train, cv=5)
y_fit = rf.predict(X_train)  # 학습 데이터 판정(4.5.3의 잘못된 방법)
y_rf = rf.predict(X_test)    # 평가 데이터에 대한 랜덤 포레스트의 판정
print(f"레이블과 일치: 학습 데이터 판정 {(y_fit == y_train).mean():.3f}, "
      f"교차검증 판정 {(y_cv == y_train).mean():.3f}")
print(f"교차검증 판정의 승인 비율 {y_cv.mean():.3f}, "
      f"기준선(모두 승인)의 충실도 {(y_rf == 1).mean():.3f}")
def tree(k):
    dt = DecisionTreeClassifier(max_depth=k, random_state=42)
    return make_pipeline(prep(), dt)
for k in range(1, 6):
    s_cv = tree(k).fit(X_train, y_cv); s_fit = tree(k).fit(X_train, y_fit)
    p = s_cv.predict(X_test)
    fid, rej = (p == y_rf).mean(), (p[y_rf == 0] == 0).mean()
    fid_fit = (s_fit.predict(X_test) == y_rf).mean()
    print(f"깊이 {k} 노드 {s_cv[-1].tree_.node_count:2d}  "
          f"충실도 {fid:.3f}  거절자 일치 {rej:.3f}  "
          f"학습 판정 트리 {fid_fit:.3f}")
    if k == 3: sur = s_cv
feat = list(sur[0].get_feature_names_out())
print(export_text(sur[-1], feature_names=feat, decimals=1,
                  show_weights=True))
print("A의 판정: 대리 트리", sur.predict(x_a)[0],
      "/ 랜덤 포레스트", rf.predict(x_a)[0])


### 더 해 보기

1. `PartialDependenceDisplay.from_estimator(rf, X_train, [("duration", "credit_amount")])`로 대출 기간과 대출 금액의 2차원 PDP를 그리고, 코드 4-4의 산점도와 겹쳐 데이터가 없는 영역을 찾아보세요.
2. 특징 하나를 빼고 랜덤 포레스트를 다시 학습해 정확도가 얼마나 떨어지는지 재어 보고(drop-column importance), 순열 중요도와 비교해 보세요.
3. 코드 4-8의 깊이를 8까지 늘려 '거절자 일치'가 어떻게 변하는지 보세요.
4. `y_train`을 `np.random.RandomState(42).permutation`으로 섞어 랜덤 포레스트를 학습하고, 순열 중요도와 불순도 중요도를 계산해 보세요(무작위화 검사).